# AISC Steel Sections and NSBA Splice Design

**Who this is for.** Steel designers who want section properties and a complete
AASHTO bolted field splice without opening a table or a spreadsheet.

**What it shows.** The one-line AISC shape database (current *and* historic
editions, with units attached), material helpers, and the NSBA-style
`design_splice` engine that turns two girder sections + loads into a fully
checked splice with article-by-article results.

**Prev:** [12. AssetWise Client](12.%20AssetWise%20Client.ipynb)
&nbsp;&middot;&nbsp; **Next:** [14. BrDR and BrM Clients ➡️](14.%20BrDR%20and%20BrM%20Clients.ipynb)

## Step 1 — Any AISC shape, one line

`W("W36X150")` gives you the full property set from the AISC database — every
value is a `pint` quantity, so units travel with the numbers and convert on
demand.

In [1]:
import os, sys
import pandas as pd

sys.path.insert(0, os.path.abspath("."))   # run from the civilpy repo root

from src.civilpy.structural.steel import W

w = W("W36X150")
print("depth        :", w.depth)
print("flange width :", w.flange_width)
print("flange thick :", w.flange_thickness)
print("web thickness:", w.web_thickness)
print("area         :", w.A)
print("Ix           :", w.I_x, " ->", w.I_x.to("ft**4"))
print("Zx           :", w.Z_x)
print("weight       :", w.W)

depth        : 35.9 inch
flange width : 12.0 inch
flange thick : 0.94 inch
web thickness: 0.625 inch
area         : 44.3 inch ** 2
Ix           : 9040.0 inch ** 4  -> 0.43595679012345684 foot ** 4
Zx           : 581.0 inch ** 3
weight       : 150.0 force_pound / foot


Every family in the manual works the same way — `W`, `M`, `S`, `HP`, `C`,
`MC`, `L`, `WT`, `MT`, `ST`, `TwoL`, `HSS`, `Pipe`:

In [2]:
from src.civilpy.structural.steel import C, HP, HSS, L, WT

shapes = [C("C15X50"), HP("HP14X117"), HSS("HSS8X8X.500"), L("L8X8X1"), WT("WT18X75")]
pd.DataFrame([{
    "shape": s.aisc_value, "weight": f"{s.weight}", "A (in²)": float(s.A.magnitude),
    "Ix (in⁴)": float(s.I_x.magnitude), "Sx (in³)": float(s.S_x.magnitude),
} for s in shapes])

,shape,weight,A (in²),Ix (in⁴),Sx (in³)
0,Type EDI_Std_Nomenclature AISC_Manual_Labe...,50.0 force_pound / foot,14.7,404.0,53.8
1,Type EDI_Std_Nomenclature AISC_Manual_Labe...,117.0 force_pound / foot,34.4,1220.0,172.0
2,Type EDI_Std_Nomenclature AISC_Manual_Lab...,48.85 force_pound / foot,13.5,125.0,31.2
3,Type EDI_Std_Nomenclature AISC_Manual_Labe...,51.0 force_pound / foot,15.1,89.1,15.8
4,Type EDI_Std_Nomenclature AISC_Manual_Labe...,75.0 force_pound / foot,22.1,698.0,53.1


## Step 2 — Historic shapes (for rating existing bridges)

Old bridges carry shapes that left the manual decades ago. `HistoricSteelSection`
searches the AISC historic editions database — wide-flange labels like `10WF19`
resolve automatically; pass `designation=` to pin a specific edition when a
label appeared in several.

In [3]:
from src.civilpy.structural.steel import HistoricSteelSection

old = HistoricSteelSection("10WF19")
print("area:", old.A, "| Ix:", old.I_x, "| Sx:", old.S_x, "| weight:", old.W)

area: 5.61 inch ** 2 | Ix: 96.2 inch ** 4 | Sx: 18.8 inch ** 3 | weight: 19.0 force_pound / foot


## Step 3 — Materials, bolts, and small utilities

In [4]:
from src.civilpy.structural.steel import (
    BoltMaterial, SteelMaterial, conv_frac_str, get_bolt_weights,
)

gr50 = SteelMaterial("A709 Gr 50", f_y=50, f_u=65)
a325 = BoltMaterial("A325", f_y=92, f_u=120)
print(gr50.__dict__, "|", a325.__dict__)

print("100 bolts, 3.5in grip, 7/8in dia, 2 washers:",
      get_bolt_weights(3.5, 0.875, 2) * 100, "lb")
print('conv_frac_str("3 7/8") =', conv_frac_str("3 7/8"), "in")

{'designation': 'A709 Gr 50', 'f_y': <Quantity(50, 'kip_per_square_inch')>, 'f_u': <Quantity(65, 'kip_per_square_inch')>, 'E': <Quantity(29000, 'kip_per_square_inch')>} | {'designation': 'A325', 'f_y': <Quantity(92, 'kip_per_square_inch')>, 'f_u': <Quantity(120, 'kip_per_square_inch')>, 'f_v': <Quantity(48, 'kip_per_square_inch')>}
100 bolts, 3.5in grip, 7/8in dia, 2 washers: 125.0 pound lb
conv_frac_str("3 7/8") = 3.875 in


## Step 4 — A bolted field splice, end to end

The NSBA/AASHTO 6.13 splice workflow in four moves:

1. **Girders** — `girder_side_from_w` builds each side straight from a rolled
   shape label (plate girders: build `GirderSide`/`Flange` by hand).
2. **Loads** — unfactored moments/shears by source (`DC1`, `DC2`, `DW`, `LL±`);
   the engine applies the Strength I / Service II factors itself.
3. **Plates** — `size_flange_splice_plates` / `size_web_splice_plate` propose
   code-minimum plate sizes you can override.
4. **`design_splice`** — bolts, plates, and every 6.13 check in one shot.

In [5]:
from src.civilpy.structural.aashto.lrfd.bolted_field_splice import (
    BoltSpec, PlatePair, SpliceInput, SpliceLoads, WebPlate,
    design_splice, girder_side_from_w,
)
from src.civilpy.structural.aashto.lrfd.splices import (
    size_flange_splice_plates, size_web_splice_plate,
)

side = girder_side_from_w("W36X150")           # same section both sides here
loads = SpliceLoads(dc1_m=700, dc1_v=45, dc2_m=150, dc2_v=12, dw_m=120, dw_v=10,
                    ll_pos_m=1350, ll_pos_v=140, ll_neg_m=-820, ll_neg_v=-95)

fp = size_flange_splice_plates(
    side.top_flange.width, side.top_flange.width, side.top_flange.thickness,
    side.web_thickness, side.web_thickness)
wp = size_web_splice_plate(33.0, side.web_thickness, side.web_thickness,
                           flange_clearance=4.0)
print("flange plates:", fp.outer_width, "x", fp.outer_thickness, "outer /",
      fp.inner_width, "x", fp.inner_thickness, "inner")
print("web plate    :", wp.height, "tall x", wp.thickness, "thick")

flange plates: 12.0 x 0.5625 outer / 5.5 x 0.625 inner
web plate    : 25.0 tall x 0.375 thick


In [6]:
plates = PlatePair("Grade 50", fp.inner_thickness, fp.inner_width,
                   fp.outer_thickness, fp.outer_width)

design = design_splice(SpliceInput(
    left=side, right=side, loads=loads,
    bolts=BoltSpec(),                          # 7/8" A325, Class B, std holes
    top_plates=plates, bottom_plates=plates,
    web_plate=WebPlate("Grade 50", wp.thickness),
))

print("everything passes:", design.ok)
design.factored_moments

everything passes: False


{'deck_cast': 0.0,
 'strength_pos': 3605.0,
 'strength_neg': -592.0,
 'service_pos': 2725.0,
 'service_neg': -96.0}

The result is three `ComponentDesign`s (top flange, bottom flange, web), each
carrying its bolt pattern, plate geometry, and a list of `CheckResult`s citing
the governing AASHTO article:

In [7]:
pd.DataFrame([{
    "component": c.name, "bolts": c.total_bolts, "rows": c.bolt_rows,
    "plate": f"{c.plate_width} x {c.plate_thickness}",
    "design force (k)": round(c.design_force, 1),
    "governing check": min(c.checks, key=lambda k: k.capacity / k.demand).name,
    "min D/C ratio": round(min(k.capacity / k.demand for k in c.checks), 2),
} for c in design.components])

,component,bolts,rows,plate,design force (k),governing check,min D/C ratio
0,top_flange,8,4,12.0 x 0.5625,424.5,inner plate net-section fracture,1.11
1,bottom_flange,8,4,12.0 x 0.5625,424.5,bottom flange Service II slip,0.33
2,web,68,2,14.75 x 0.375,616.6,web plate shear rupture,-0.15


In [8]:
pd.DataFrame([{
    "article": k.article, "check": k.name,
    "demand": round(k.demand, 1), "capacity": round(k.capacity, 1),
    "D/C": round(k.demand / k.capacity, 3),
} for k in design.top_flange.checks])

,article,check,demand,capacity,D/C
0,6.13.2.8,top flange Service II slip,33.0,312.0,0.106
1,6.13.5.2,outer plate yield (tension),212.2,320.6,0.662
2,6.13.5.2,Net Section Reduction Limit,4.6,5.7,0.809
3,6.13.6.1.3,outer plate net-section fracture,212.2,241.3,0.880
4,6.13.4,outer plate block shear,212.2,284.3,0.746
5,6.13.5.2,inner plate yield (tension),212.2,326.6,0.650
6,6.13.5.2,Net Section Reduction Limit,4.5,5.8,0.775
7,6.13.6.1.3,inner plate net-section fracture,212.2,235.6,0.901
8,6.13.4,inner plate block shear,212.2,315.9,0.672
9,6.13.4,girder flange block shear (Mode 1),424.5,507.3,0.837


## Step 5 — The primitives are yours too

Everything `design_splice` composes is importable on its own from
`civilpy.structural.aashto.lrfd.splices` — design forces (6.13.6.1.3), plate
sizing, filler reductions, spacing/edge limits — when you need one number, not
a whole splice:

In [9]:
from src.civilpy.structural.aashto.lrfd.splices import bolt_spacing_limits

bolt_spacing_limits(d_bolt=0.875, plate_t=0.5625)

SpacingLimits(min_spacing=2.625, max_spacing_seal=6.25, min_edge=1.125, max_edge=4.5, pitch_ok=None, gage_ok=None, edge_ok=None, end_ok=None)

**Deeper dives:** the named notebooks *Bolted Field Splice Design Process*
(step-by-step hand calc), *nsba_splice_check* / *nsba_splice_validation*
(verification against the NSBA worked example), and *AISC Steel Specs*.